In [23]:
from pathlib import Path
import ast
import pandas as pd
from collections import Counter

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "mabsa"
RESULTS_DIR = PROJECT_ROOT / "results" / "eda"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Data directory:", DATA_DIR.resolve())
print("Results directory:", RESULTS_DIR.resolve())

Project root: D:\projects\J26-DS-349\component_1
Data directory: D:\projects\J26-DS-349\component_1\data\raw\mabsa
Results directory: D:\projects\J26-DS-349\component_1\results\eda


In [24]:
expected_files = []

for domain in ["phone", "laptop"]:
    for split in ["train", "dev", "test"]:
        path = DATA_DIR / domain / "en" / f"{split}.txt"
        expected_files.append(path)

for path in expected_files:
    print(path, "->", "FOUND" if path.exists() else "MISSING")

..\data\raw\mabsa\phone\en\train.txt -> FOUND
..\data\raw\mabsa\phone\en\dev.txt -> FOUND
..\data\raw\mabsa\phone\en\test.txt -> FOUND
..\data\raw\mabsa\laptop\en\train.txt -> FOUND
..\data\raw\mabsa\laptop\en\dev.txt -> FOUND
..\data\raw\mabsa\laptop\en\test.txt -> FOUND


In [25]:
sample_file = DATA_DIR / "phone" / "en" / "train.txt"

with open(sample_file, "r", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline().strip())
        print("-" * 100)

Bought this as a quick buy for my son the day he smashed his Huawei. Didn't want to buy anything too expensive for the same above reason. He is very happy with it. The resolution is amazing and it's super fast. The screen size is a perfect fit in the palm of the hand but he uses it popsocket for a better grip. Memory is huge which is a benefit too. Phone screen protection is on already and phone case cover is also included. Good bargain####[['happy with', 'Overall#Overall', 'positive'], ['resolution', 'Screen#Clarity', 'positive'], ['screen size', 'Screen#Size', 'positive'], ['Memory', 'Product Configuration#Operating Memory', 'positive'], ['Memory', 'Product Configuration#Operating Memory', 'positive'], ['Phone screen protection', 'Product Accessories#Cell Phone Film', 'positive'], ['phone case cover', 'Product Accessories#Phone Cases', 'positive'], ['bargain', 'Price#Price', 'positive']]
-------------------------------------------------------------------------------------------------

In [26]:
def parse_mabsa_line(line):
    line = line.strip()

    if not line:
        return None

    if "####" not in line:
        return {
            "text": line,
            "triplets": [],
            "parse_status": "no_separator"
        }

    text, label_text = line.split("####", 1)

    try:
        triplets = ast.literal_eval(label_text)
        status = "ok"
    except Exception:
        triplets = []
        status = "label_parse_error"

    return {
        "text": text.strip(),
        "triplets": triplets,
        "parse_status": status
    }

In [27]:
with open(sample_file, "r", encoding="utf-8") as f:
    first_line = f.readline()

parse_mabsa_line(first_line)

{'text': "Bought this as a quick buy for my son the day he smashed his Huawei. Didn't want to buy anything too expensive for the same above reason. He is very happy with it. The resolution is amazing and it's super fast. The screen size is a perfect fit in the palm of the hand but he uses it popsocket for a better grip. Memory is huge which is a benefit too. Phone screen protection is on already and phone case cover is also included. Good bargain",
 'triplets': [['happy with', 'Overall#Overall', 'positive'],
  ['resolution', 'Screen#Clarity', 'positive'],
  ['screen size', 'Screen#Size', 'positive'],
  ['Memory', 'Product Configuration#Operating Memory', 'positive'],
  ['Memory', 'Product Configuration#Operating Memory', 'positive'],
  ['Phone screen protection',
   'Product Accessories#Cell Phone Film',
   'positive'],
  ['phone case cover', 'Product Accessories#Phone Cases', 'positive'],
  ['bargain', 'Price#Price', 'positive']],
 'parse_status': 'ok'}

In [28]:
def load_split(domain, split):
    file_path = DATA_DIR / domain / "en" / f"{split}.txt"

    rows = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            parsed = parse_mabsa_line(line)

            if parsed is None:
                continue

            rows.append({
                "domain": domain,
                "split": split,
                "line_number": line_number,
                "text": parsed["text"],
                "triplets": parsed["triplets"],
                "parse_status": parsed["parse_status"],
            })

    return pd.DataFrame(rows)


In [29]:
dfs = []

for domain in ["phone", "laptop"]:
    for split in ["train", "dev", "test"]:
        dfs.append(load_split(domain, split))

df = pd.concat(dfs, ignore_index=True)

print("Combined dataset shape:", df.shape)

df.head()

Combined dataset shape: (4227, 6)


,domain,split,line_number,text,triplets,parse_status
0,phone,train,1,Bought this as a quick buy for my son the day ...,"[[happy with, Overall#Overall, positive], [res...",ok
1,phone,train,2,Camera quality not well. battery life not as p...,"[[Camera quality, Camera#General, negative], [...",ok
2,phone,train,3,Your browser does not support HTML5 video. ?Vf...,"[[browser, System#Software Compatibility, nega...",ok
3,phone,train,4,Received item as described. Delivered from ove...,"[[Received item, Logistics#Lost and Damaged, p...",ok
4,phone,train,5,Item as described. Fast delivery. Have purchas...,"[[delivery, Logistics#Speed, positive]]",ok


In [30]:
dataset_summary = (
    df.groupby(["domain", "split"])
      .size()
      .reset_index(name="num_sentences")
)

dataset_summary
dataset_summary.to_csv(
    RESULTS_DIR / "dataset_summary.csv",
    index=False
)

In [31]:
df["parse_status"].value_counts()

parse_status
ok    4227
Name: count, dtype: int64

In [32]:
df["num_triplets"] = df["triplets"].apply(len)

df.groupby("domain")["num_triplets"].describe()


,count,mean,std,min,25%,50%,75%,max
domain,,,,,,,,
laptop,2122.0,1.392554,0.766459,1.0,1.0,1.0,2.0,8.0
phone,2105.0,2.285036,1.836856,0.0,1.0,2.0,3.0,13.0


In [33]:
empty_labels = (
    df.assign(no_label=df["num_triplets"] == 0)
      .groupby("domain")["no_label"]
      .agg(["sum", "mean"])
)

empty_labels

,sum,mean
domain,,
laptop,0,0.000000
phone,327,0.155344


In [34]:
triplet_rows = []

for _, row in df.iterrows():
    for triplet in row["triplets"]:
        if len(triplet) != 3:
            continue

        aspect_term, aspect_category, sentiment = triplet

        triplet_rows.append({
            "domain": row["domain"],
            "split": row["split"],
            "text": row["text"],
            "aspect_term": aspect_term,
            "aspect_category": aspect_category,
            "sentiment": sentiment
        })

triplets_df = pd.DataFrame(triplet_rows)

print(triplets_df.shape)
triplets_df.head()

(7765, 6)


,domain,split,text,aspect_term,aspect_category,sentiment
0,phone,train,Bought this as a quick buy for my son the day ...,happy with,Overall#Overall,positive
1,phone,train,Bought this as a quick buy for my son the day ...,resolution,Screen#Clarity,positive
2,phone,train,Bought this as a quick buy for my son the day ...,screen size,Screen#Size,positive
3,phone,train,Bought this as a quick buy for my son the day ...,Memory,Product Configuration#Operating Memory,positive
4,phone,train,Bought this as a quick buy for my son the day ...,Memory,Product Configuration#Operating Memory,positive


In [35]:
phone_aspects = sorted(
    triplets_df.loc[
        triplets_df["domain"] == "phone",
        "aspect_category"
    ].dropna().unique()
)

laptop_aspects = sorted(
    triplets_df.loc[
        triplets_df["domain"] == "laptop",
        "aspect_category"
    ].dropna().unique()
)

print("PHONE ASPECT CATEGORIES:")
for x in phone_aspects:
    print(x)

print("\nLAPTOP ASPECT CATEGORIES:")
for x in laptop_aspects:
    print(x)

PHONE ASPECT CATEGORIES:
After-sales Service#Exchange/Warranty/Return
Appearance Design#Aesthetics General
Appearance Design#Color
Appearance Design#Exterior Design Material
Appearance Design#Fuselage Size
Appearance Design#Grip Feeling
Appearance Design#Thickness
Appearance Design#Weight
Appearance Design#Workmanship and Texture
Audio/Sound#Tone quality
Audio/Sound#Volume and Speaker
Battery/Longevity#Battery Capacity
Battery/Longevity#Battery Life
Battery/Longevity#Charging Method
Battery/Longevity#Charging Speed
Battery/Longevity#General
Battery/Longevity#Power Consumption Speed
Battery/Longevity#Standby Time
Branding/Marketing#Promotional Giveaways
Buyer Attitude#Loyalty
Buyer Attitude#Recommendable
Buyer Attitude#Repurchase and Churn Tendency
Buyer Attitude#Shopping Experiences
Buyer Attitude#Shopping Willingness
Camera#Fill light
Camera#Front Camera
Camera#General
Camera#Rear Camera
Ease of Use#Audience Groups
Ease of Use#Easy to Use
Intelligent Assistant#Intelligent Assistant Ge

In [36]:
sorted(triplets_df["sentiment"].dropna().unique())

sentiment_summary = (
    triplets_df.groupby(["domain", "sentiment"])
               .size()
               .reset_index(name="count")
)

sentiment_summary

,domain,sentiment,count
0,laptop,negative,921
1,laptop,neutral,177
2,laptop,positive,1857
3,phone,negative,1057
4,phone,neutral,198
5,phone,positive,3555


In [37]:
sentiment_summary.to_csv(
    RESULTS_DIR / "sentiment_summary.csv",
    index=False
)

In [38]:
aspect_summary = (
    triplets_df.groupby(["domain", "aspect_category"])
               .size()
               .reset_index(name="count")
               .sort_values(
                   ["domain", "count"],
                   ascending=[True, False]
               )
)

aspect_summary

,domain,aspect_category,count
48,laptop,LAPTOP#GENERAL,624
50,laptop,LAPTOP#OPERATION_PERFORMANCE,319
47,laptop,LAPTOP#DESIGN_FEATURES,243
52,laptop,LAPTOP#PRICE,116
53,laptop,LAPTOP#QUALITY,104
...,...,...,...
132,phone,Camera#Fill light,2
163,phone,Product Quality#Cleanliness,2
180,phone,Signal#Call Quality,2
141,phone,Logistics#Shipping Fee,1


In [39]:
aspect_summary.to_csv(
    RESULTS_DIR / "aspect_summary.csv",
    index=False
)

In [40]:
multi_aspect_summary = (
    df.assign(
        is_multi_aspect=df["num_triplets"] > 1
    )
    .groupby("domain")
    .agg(
        total_reviews=("text", "count"),
        multi_aspect_reviews=("is_multi_aspect", "sum"),
        multi_aspect_rate=("is_multi_aspect", "mean"),
    )
)

multi_aspect_summary

,total_reviews,multi_aspect_reviews,multi_aspect_rate
domain,,,
laptop,2122,592,0.278982
phone,2105,1340,0.636580


In [41]:
null_aspects = triplets_df[
    triplets_df["aspect_term"].astype(str).str.upper() == "NULL"
]

print("NULL aspect-term count:", len(null_aspects))
null_aspects.head(20)

NULL aspect-term count: 627


,domain,split,text,aspect_term,aspect_category,sentiment
4811,laptop,train,i bought this as a gift and was incredibly dis...,NULL,LAPTOP#GENERAL,negative
4815,laptop,train,the last straw happened recently when i put it...,NULL,LAPTOP#OPERATION_PERFORMANCE,negative
4827,laptop,train,it ' s so easy to travel with .,NULL,LAPTOP#PORTABILITY,positive
4829,laptop,train,overall i ' m happy with it for what i use it ...,NULL,LAPTOP#GENERAL,positive
4830,laptop,train,but it lost the coil whine roulette - - badly .,NULL,LAPTOP#DESIGN_FEATURES,negative
4835,laptop,train,it is far and away the best i ' ve ever had .,NULL,LAPTOP#GENERAL,positive
4851,laptop,train,that is just incredible .,NULL,LAPTOP#GENERAL,positive
4855,laptop,train,"pros : nice size , clear screen , quick on sta...",NULL,LAPTOP#DESIGN_FEATURES,positive
4857,laptop,train,"pros : nice size , clear screen , quick on sta...",NULL,LAPTOP#OPERATION_PERFORMANCE,positive
4858,laptop,train,"pros : nice size , clear screen , quick on sta...",NULL,LAPTOP#USABILITY,positive


In [42]:
print("=" * 60)
print("1. DATASET SUMMARY")
print("=" * 60)
print(dataset_summary.to_string(index=False))


print("\n" + "=" * 60)
print("2. PHONE ASPECT CATEGORIES")
print("=" * 60)

for aspect in phone_aspects:
    print(aspect)


print("\n" + "=" * 60)
print("3. LAPTOP ASPECT CATEGORIES")
print("=" * 60)

for aspect in laptop_aspects:
    print(aspect)


print("\n" + "=" * 60)
print("4. SENTIMENT LABELS")
print("=" * 60)

for label in sentiment_labels:
    print(label)


print("\n" + "=" * 60)
print("5. NULL ASPECT-TERM COUNT")
print("=" * 60)

print(len(null_aspects))


print("\n" + "=" * 60)
print("6. MULTI-ASPECT SUMMARY")
print("=" * 60)

print(multi_aspect_summary)

1. DATASET SUMMARY
domain split  num_sentences
laptop   dev            326
laptop  test            532
laptop train           1264
 phone   dev            307
 phone  test            522
 phone train           1276

2. PHONE ASPECT CATEGORIES
After-sales Service#Exchange/Warranty/Return
Appearance Design#Aesthetics General
Appearance Design#Color
Appearance Design#Exterior Design Material
Appearance Design#Fuselage Size
Appearance Design#Grip Feeling
Appearance Design#Thickness
Appearance Design#Weight
Appearance Design#Workmanship and Texture
Audio/Sound#Tone quality
Audio/Sound#Volume and Speaker
Battery/Longevity#Battery Capacity
Battery/Longevity#Battery Life
Battery/Longevity#Charging Method
Battery/Longevity#Charging Speed
Battery/Longevity#General
Battery/Longevity#Power Consumption Speed
Battery/Longevity#Standby Time
Branding/Marketing#Promotional Giveaways
Buyer Attitude#Loyalty
Buyer Attitude#Recommendable
Buyer Attitude#Repurchase and Churn Tendency
Buyer Attitude#Shopping 

NameError: name 'sentiment_labels' is not defined

In [43]:
null_summary = (
    triplets_df.assign(
        is_null=triplets_df["aspect_term"].astype(str).str.upper().eq("NULL")
    )
    .groupby("domain")
    .agg(
        total_triplets=("aspect_term", "count"),
        null_aspect_terms=("is_null", "sum"),
        null_rate=("is_null", "mean"),
    )
)

null_summary

,total_triplets,null_aspect_terms,null_rate
domain,,,
laptop,2955,627,0.212183
phone,4810,0,0.000000


In [44]:
null_summary.to_csv(
    RESULTS_DIR / "null_aspect_summary.csv",
    index=True
)

In [45]:
triplets_df[
    triplets_df["aspect_term"].astype(str).str.upper().eq("NULL")
][
    ["domain", "text", "aspect_category", "sentiment"]
].head(30)

,domain,text,aspect_category,sentiment
4811,laptop,i bought this as a gift and was incredibly dis...,LAPTOP#GENERAL,negative
4815,laptop,the last straw happened recently when i put it...,LAPTOP#OPERATION_PERFORMANCE,negative
4827,laptop,it ' s so easy to travel with .,LAPTOP#PORTABILITY,positive
4829,laptop,overall i ' m happy with it for what i use it ...,LAPTOP#GENERAL,positive
4830,laptop,but it lost the coil whine roulette - - badly .,LAPTOP#DESIGN_FEATURES,negative
4835,laptop,it is far and away the best i ' ve ever had .,LAPTOP#GENERAL,positive
4851,laptop,that is just incredible .,LAPTOP#GENERAL,positive
4855,laptop,"pros : nice size , clear screen , quick on sta...",LAPTOP#DESIGN_FEATURES,positive
4857,laptop,"pros : nice size , clear screen , quick on sta...",LAPTOP#OPERATION_PERFORMANCE,positive
4858,laptop,"pros : nice size , clear screen , quick on sta...",LAPTOP#USABILITY,positive


In [46]:
aspect_frequency = (
    triplets_df.groupby(["domain", "aspect_category"])
    .size()
    .reset_index(name="count")
    .sort_values(["domain", "count"], ascending=[True, False])
)

aspect_frequency

,domain,aspect_category,count
48,laptop,LAPTOP#GENERAL,624
50,laptop,LAPTOP#OPERATION_PERFORMANCE,319
47,laptop,LAPTOP#DESIGN_FEATURES,243
52,laptop,LAPTOP#PRICE,116
53,laptop,LAPTOP#QUALITY,104
...,...,...,...
132,phone,Camera#Fill light,2
163,phone,Product Quality#Cleanliness,2
180,phone,Signal#Call Quality,2
141,phone,Logistics#Shipping Fee,1


In [47]:
aspect_frequency[
    aspect_frequency["domain"] == "phone"
].head(30)

,domain,aspect_category,count
144,phone,Overall#Overall,910
142,phone,Logistics#Speed,402
140,phone,Logistics#Lost and Damaged,256
128,phone,Buyer Attitude#Recommendable,193
148,phone,Price#Price,150
173,phone,Seller Service#Attitude,149
134,phone,Camera#General,145
166,phone,Product Quality#General,123
120,phone,Battery/Longevity#Battery Life,105
177,phone,Seller Service#Timeliness of Seller Service,103


In [48]:
aspect_frequency[
    aspect_frequency["domain"] == "laptop"
].head(30)

,domain,aspect_category,count
48,laptop,LAPTOP#GENERAL,624
50,laptop,LAPTOP#OPERATION_PERFORMANCE,319
47,laptop,LAPTOP#DESIGN_FEATURES,243
52,laptop,LAPTOP#PRICE,116
53,laptop,LAPTOP#QUALITY,104
2,laptop,BATTERY#OPERATION_PERFORMANCE,76
39,laptop,KEYBOARD#GENERAL,75
14,laptop,DISPLAY#GENERAL,74
13,laptop,DISPLAY#DESIGN_FEATURES,67
54,laptop,LAPTOP#USABILITY,67


In [49]:
aspect_frequency.to_csv(
    RESULTS_DIR / "aspect_frequency.csv",
    index=False
)

In [50]:
phone_mapping = pd.DataFrame({
    "domain": "phone",
    "source_label": phone_aspects
})

laptop_mapping = pd.DataFrame({
    "domain": "laptop",
    "source_label": laptop_aspects
})

taxonomy_mapping = pd.concat(
    [phone_mapping, laptop_mapping],
    ignore_index=True
)

taxonomy_mapping["canonical_aspect"] = ""
taxonomy_mapping["semantic_group"] = ""
taxonomy_mapping["include"] = ""
taxonomy_mapping["mapping_status"] = "pending"
taxonomy_mapping["notes"] = ""

taxonomy_mapping.head()

,domain,source_label,canonical_aspect,semantic_group,include,mapping_status,notes
0,phone,After-sales Service#Exchange/Warranty/Return,,,,pending,
1,phone,Appearance Design#Aesthetics General,,,,pending,
2,phone,Appearance Design#Color,,,,pending,
3,phone,Appearance Design#Exterior Design Material,,,,pending,
4,phone,Appearance Design#Fuselage Size,,,,pending,


In [51]:
taxonomy_mapping.to_csv(
    PROJECT_ROOT / "taxonomy" / "mabsa_taxonomy_mapping_v0_1.csv",
    index=False
)

print("Mapping template saved.")

Mapping template saved.


In [52]:
# ============================================================
# BUILD M-ABSA TAXONOMY MAPPING TEMPLATE
# ============================================================

# All unique source labels
all_labels = (
    triplets_df[["domain", "aspect_category"]]
    .drop_duplicates()
    .rename(columns={"aspect_category": "source_label"})
)

# Add observed frequency
mapping_df = all_labels.merge(
    aspect_frequency.rename(
        columns={"aspect_category": "source_label"}
    ),
    on=["domain", "source_label"],
    how="left"
)

In [ ]:
def candidate_taxonomy_mapping(domain, label):
    label_upper = label.upper()

    # ========================================================
    # COMMON ASPECTS
    # ========================================================

    # Battery
    if (
        "BATTERY/LONGEVITY#BATTERY LIFE" in label_upper
        or "BATTERY/LONGEVITY#BATTERY CAPACITY" in label_upper
        or label_upper.startswith("BATTERY#")
    ):
        return "battery", "COMMON", "yes", "candidate"

    # Charging
    if (
        "BATTERY/LONGEVITY#CHARGING" in label_upper
        or label_upper.startswith("POWER_SUPPLY#")
    ):
        return "charging", "COMMON", "review", "candidate"

    # Display
    if (
        label_upper.startswith("SCREEN#")
        or label_upper.startswith("DISPLAY#")
    ):
        return "display", "COMMON", "yes", "candidate"

    # Price
    if (
        label_upper.startswith("PRICE#")
        or label_upper == "LAPTOP#PRICE"
    ):
        return "price", "COMMON", "yes", "candidate"

    # Performance
    if (
        label_upper.startswith("PERFORMANCE#")
        or label_upper == "SYSTEM#OPERATION SMOOTHNESS"
        or label_upper == "LAPTOP#OPERATION_PERFORMANCE"
    ):
        return "performance", "COMMON", "yes", "candidate"

    # Design
    if (
        label_upper.startswith("APPEARANCE DESIGN#")
        or label_upper == "LAPTOP#DESIGN_FEATURES"
    ):
        return "design", "COMMON", "yes", "candidate"

    # CPU
    if (
        label_upper == "PRODUCT CONFIGURATION#CPU"
        or label_upper.startswith("CPU#")
    ):
        return "cpu", "COMMON", "yes", "candidate"

    # Memory
    if (
        label_upper in {
            "PRODUCT CONFIGURATION#MEMORY",
            "PRODUCT CONFIGURATION#OPERATING MEMORY",
        }
        or label_upper.startswith("MEMORY#")
    ):
        return "memory", "COMMON", "yes", "candidate"

    # ========================================================
    # PHONE-SPECIFIC
    # ========================================================

    if label_upper.startswith("CAMERA#"):
        return "camera", "PHONE_SPECIFIC", "yes", "candidate"

    if label_upper.startswith("SHOOTING FUNCTIONS#"):
        return "camera", "PHONE_SPECIFIC", "yes", "candidate"

    if label_upper.startswith("SIGNAL#"):
        return "signal_network", "PHONE_SPECIFIC", "yes", "candidate"

    if label_upper == "SECURITY#SCREEN UNLOCK":
        return "biometrics_security", "PHONE_SPECIFIC", "yes", "candidate"

    if label_upper == "SMART CONNECT#POSITIONING AND GPS":
        return "gps", "PHONE_SPECIFIC", "yes", "candidate"

    if label_upper == "SYSTEM#NFC":
        return "nfc", "PHONE_SPECIFIC", "yes", "candidate"

    # ========================================================
    # LAPTOP-SPECIFIC
    # ========================================================

    if label_upper.startswith("KEYBOARD#"):
        return "keyboard", "LAPTOP_SPECIFIC", "yes", "candidate"

    if label_upper.startswith("PORTS#"):
        return "ports", "LAPTOP_SPECIFIC", "yes", "candidate"

    if label_upper.startswith("FANS&COOLING#"):
        return "fans_cooling", "LAPTOP_SPECIFIC", "yes", "candidate"

    if label_upper.startswith("HARD_DISC#"):
        return "storage", "LAPTOP_SPECIFIC", "yes", "candidate"

    if label_upper.startswith("MOUSE#"):
        return "mouse", "LAPTOP_SPECIFIC", "review", "candidate"

    if label_upper.startswith("MOTHERBOARD#"):
        return "motherboard", "LAPTOP_SPECIFIC", "yes", "candidate"

    if label_upper.startswith("OPTICAL_DRIVES#"):
        return "optical_drive", "LAPTOP_SPECIFIC", "yes", "candidate"

    # ========================================================
    # CLEAR EXCLUSIONS / OUT-OF-SCOPE
    # ========================================================

    if (
        label_upper.startswith("BUYER ATTITUDE#")
        or label_upper.startswith("LOGISTICS#")
        or label_upper.startswith("SELLER SERVICE#")
        or label_upper.startswith("BRANDING/MARKETING#")
        or label_upper.startswith("PRODUCT PACKAGING#")
        or label_upper.startswith("AFTER-SALES SERVICE#")
        or label_upper.startswith("OUT_OF_SCOPE#")
        or label_upper.startswith("SHIPPING#")
        or label_upper.startswith("SUPPORT#")
        or label_upper.startswith("COMPANY#")
        or label_upper.startswith("WARRANTY#")
        or label_upper == "OVERALL#OVERALL"
        or label_upper == "LAPTOP#GENERAL"
    ):
        return "", "", "no", "candidate"


    return "", "", "review", "pending"

In [54]:
mapped_values = mapping_df.apply(
    lambda row: candidate_taxonomy_mapping(
        row["domain"],
        row["source_label"]
    ),
    axis=1
)

mapping_df[
    [
        "canonical_aspect",
        "semantic_group",
        "include",
        "mapping_status"
    ]
] = pd.DataFrame(
    mapped_values.tolist(),
    index=mapping_df.index
)

mapping_df.head(30)

,domain,source_label,count,canonical_aspect,semantic_group,include,mapping_status
0,phone,Overall#Overall,910,,,no,candidate
1,phone,Screen#Clarity,22,display,COMMON,yes,candidate
2,phone,Screen#Size,35,display,COMMON,yes,candidate
3,phone,Product Configuration#Operating Memory,35,memory,COMMON,yes,candidate
4,phone,Product Accessories#Cell Phone Film,46,,,review,pending
5,phone,Product Accessories#Phone Cases,69,,,review,pending
6,phone,Price#Price,150,price,COMMON,yes,candidate
7,phone,Camera#General,145,camera,PHONE_SPECIFIC,yes,candidate
8,phone,Battery/Longevity#Battery Life,105,battery,COMMON,yes,candidate
9,phone,System#Software Compatibility,30,,,review,pending


In [55]:
mapping_path = (
    PROJECT_ROOT
    / "taxonomy"
    / "mabsa_taxonomy_mapping_v0_1.csv"
)

mapping_df.to_csv(
    mapping_path,
    index=False
)

print("Saved taxonomy mapping to:")
print(mapping_path.resolve())

Saved taxonomy mapping to:
D:\projects\J26-DS-349\component_1\taxonomy\mabsa_taxonomy_mapping_v0_1.csv


In [56]:
print("=" * 70)
print("MAPPING STATUS SUMMARY")
print("=" * 70)

print(
    mapping_df.groupby(
        ["domain", "include"]
    ).size()
)

print("\n" + "=" * 70)
print("SEMANTIC GROUP SUMMARY")
print("=" * 70)

print(
    mapping_df[
        mapping_df["include"] == "yes"
    ]
    .groupby(
        ["domain", "semantic_group"]
    )
    .size()
)

MAPPING STATUS SUMMARY
domain  include
laptop  no         18
        review     40
        yes        50
phone   no         22
        review     29
        yes        35
dtype: int64

SEMANTIC GROUP SUMMARY
domain  semantic_group 
laptop  COMMON             22
        LAPTOP_SPECIFIC    28
phone   COMMON             22
        PHONE_SPECIFIC     13
dtype: int64


In [57]:
review_labels = mapping_df[
    mapping_df["include"] == "review"
][
    [
        "domain",
        "source_label",
        "count",
        "canonical_aspect",
        "semantic_group",
        "mapping_status"
    ]
]

print(
    review_labels.to_string(index=False)
)

domain                                        source_label  count canonical_aspect  semantic_group mapping_status
 phone                 Product Accessories#Cell Phone Film     46                                         pending
 phone                     Product Accessories#Phone Cases     69                                         pending
 phone                       System#Software Compatibility     30                                         pending
 phone                           Battery/Longevity#General     84                                         pending
 phone                                  System#Application     32                                         pending
 phone                             Ease of Use#Easy to Use     84                                         pending
 phone           Battery/Longevity#Power Consumption Speed     32                                         pending
 phone                      Product Accessories#Headphones     40                       

In [58]:
# TARGETED EXAMPLE INSPECTION FOR AMBIGUOUS TAXONOMY LABELS

labels_to_inspect = [
    # ---------------- PHONE ----------------
    ("phone", "Overall#Overall"),
    ("phone", "Battery/Longevity#General"),
    ("phone", "Battery/Longevity#Power Consumption Speed"),
    ("phone", "Battery/Longevity#Standby Time"),
    ("phone", "Ease of Use#Easy to Use"),
    ("phone", "System#System General"),
    ("phone", "System#Application"),
    ("phone", "System#Software Compatibility"),
    ("phone", "System#System Upgrade"),
    ("phone", "System#UI Interface Aesthetics"),
    ("phone", "Audio/Sound#Tone quality"),
    ("phone", "Audio/Sound#Volume and Speaker"),
    ("phone", "Smart Connect#Bluetooth Connection"),
    ("phone", "Product Quality#General"),
    ("phone", "Product Quality#Fall Protection"),
    ("phone", "Product Quality#Dustproof"),
    ("phone", "Product Quality#Water Resistant"),

    # ---------------- LAPTOP ----------------
    ("laptop", "LAPTOP#GENERAL"),
    ("laptop", "LAPTOP#QUALITY"),
    ("laptop", "LAPTOP#USABILITY"),
    ("laptop", "LAPTOP#PORTABILITY"),
    ("laptop", "POWER_SUPPLY#GENERAL"),
    ("laptop", "POWER_SUPPLY#OPERATION_PERFORMANCE"),
    ("laptop", "MULTIMEDIA_DEVICES#GENERAL"),
    ("laptop", "MULTIMEDIA_DEVICES#OPERATION_PERFORMANCE"),
    ("laptop", "OS#GENERAL"),
    ("laptop", "OS#OPERATION_PERFORMANCE"),
    ("laptop", "SOFTWARE#GENERAL"),
    ("laptop", "SOFTWARE#OPERATION_PERFORMANCE"),
    ("laptop", "HARDWARE#GENERAL"),
    ("laptop", "GRAPHICS#GENERAL"),
    ("laptop", "LAPTOP#CONNECTIVITY"),
    ("laptop", "MOUSE#GENERAL"),
]


def show_examples(domain, label, n=3):
    subset = triplets_df[
        (triplets_df["domain"] == domain) &
        (triplets_df["aspect_category"] == label)
    ]

    print("\n" + "=" * 90)
    print(f"{domain.upper()} | {label}")
    print(f"Total annotations: {len(subset)}")
    print("=" * 90)

    if len(subset) == 0:
        print("No examples found.")
        return

    sample = subset[
        ["text", "aspect_term", "sentiment"]
    ].head(n)

    for i, row in enumerate(sample.itertuples(index=False), start=1):
        print(f"\nExample {i}")
        print("Text       :", row.text)
        print("Aspect term:", row.aspect_term)
        print("Sentiment  :", row.sentiment)


for domain, label in labels_to_inspect:
    show_examples(domain, label, n=3)


PHONE | Overall#Overall
Total annotations: 910

Example 1
Text       : Bought this as a quick buy for my son the day he smashed his Huawei. Didn't want to buy anything too expensive for the same above reason. He is very happy with it. The resolution is amazing and it's super fast. The screen size is a perfect fit in the palm of the hand but he uses it popsocket for a better grip. Memory is huge which is a benefit too. Phone screen protection is on already and phone case cover is also included. Good bargain
Aspect term: happy with
Sentiment  : positive

Example 2
Text       : No The product is great No not possible to play Cod or free fire on this
Aspect term: product
Sentiment  : positive

Example 3
Text       : Tested by repairing Nintendo switch joycon controllers, works well
Aspect term: works
Sentiment  : positive

PHONE | Battery/Longevity#General
Total annotations: 84

Example 1
Text       : After half a year usage,I can confirm its a dodge spy phone! there's intermittent faults

In [59]:
for domain, label in labels_to_inspect:
    if domain == "phone":
        show_examples(domain, label, n=3)


PHONE | Overall#Overall
Total annotations: 910

Example 1
Text       : Bought this as a quick buy for my son the day he smashed his Huawei. Didn't want to buy anything too expensive for the same above reason. He is very happy with it. The resolution is amazing and it's super fast. The screen size is a perfect fit in the palm of the hand but he uses it popsocket for a better grip. Memory is huge which is a benefit too. Phone screen protection is on already and phone case cover is also included. Good bargain
Aspect term: happy with
Sentiment  : positive

Example 2
Text       : No The product is great No not possible to play Cod or free fire on this
Aspect term: product
Sentiment  : positive

Example 3
Text       : Tested by repairing Nintendo switch joycon controllers, works well
Aspect term: works
Sentiment  : positive

PHONE | Battery/Longevity#General
Total annotations: 84

Example 1
Text       : After half a year usage,I can confirm its a dodge spy phone! there's intermittent faults

In [60]:
for domain, label in labels_to_inspect:
    if domain == "laptop":
        show_examples(domain, label, n=3)


LAPTOP | LAPTOP#GENERAL
Total annotations: 624

Example 1
Text       : i bought this as a gift and was incredibly disappointed as it did n ' t even turn on after the initial charging .
Aspect term: NULL
Sentiment  : negative

Example 2
Text       : i purchased this item 7 months ago and i love it .
Aspect term: item
Sentiment  : positive

Example 3
Text       : i ' ve had the laptop for a full day now and i can say it is quite impressive .
Aspect term: laptop
Sentiment  : positive

LAPTOP | LAPTOP#QUALITY
Total annotations: 104

Example 1
Text       : it ' s a solid and quality built laptop .
Aspect term: laptop
Sentiment  : positive

Example 2
Text       : it ' s a solid and quality built laptop .
Aspect term: laptop
Sentiment  : positive

Example 3
Text       : * very solidly built and it transitions nicely from laptop to tablet mode .
Aspect term: built
Sentiment  : positive

LAPTOP | LAPTOP#USABILITY
Total annotations: 67

Example 1
Text       : it makes the entire device unusable

In [61]:
mapped_values = mapping_df.apply(
    lambda row: candidate_taxonomy_mapping(
        row["domain"],
        row["source_label"]
    ),
    axis=1
)

mapping_df[
    [
        "canonical_aspect",
        "semantic_group",
        "include",
        "mapping_status"
    ]
] = pd.DataFrame(
    mapped_values.tolist(),
    index=mapping_df.index
)

mapping_df.head(20)

,domain,source_label,count,canonical_aspect,semantic_group,include,mapping_status
0,phone,Overall#Overall,910,,,no,candidate
1,phone,Screen#Clarity,22,display,COMMON,yes,candidate
2,phone,Screen#Size,35,display,COMMON,yes,candidate
3,phone,Product Configuration#Operating Memory,35,memory,COMMON,yes,candidate
4,phone,Product Accessories#Cell Phone Film,46,,,review,pending
5,phone,Product Accessories#Phone Cases,69,,,review,pending
6,phone,Price#Price,150,price,COMMON,yes,candidate
7,phone,Camera#General,145,camera,PHONE_SPECIFIC,yes,candidate
8,phone,Battery/Longevity#Battery Life,105,battery,COMMON,yes,candidate
9,phone,System#Software Compatibility,30,,,review,pending


In [62]:
print(
    "Overall#Overall →",
    candidate_taxonomy_mapping(
        "phone",
        "Overall#Overall"
    )
)

print(
    "Camera#General →",
    candidate_taxonomy_mapping(
        "phone",
        "Camera#General"
    )
)

Overall#Overall → ('', '', 'no', 'candidate')
Camera#General → ('camera', 'PHONE_SPECIFIC', 'yes', 'candidate')


In [63]:
phone_labels_to_inspect = [
    "Battery/Longevity#General",
    "Battery/Longevity#Power Consumption Speed",
    "Battery/Longevity#Standby Time",
    "Ease of Use#Easy to Use",
    "System#System General",
    "System#Application",
    "System#Software Compatibility",
    "System#System Upgrade",
    "System#UI Interface Aesthetics",
    "Audio/Sound#Tone quality",
    "Audio/Sound#Volume and Speaker",
    "Smart Connect#Bluetooth Connection",
    "Product Quality#General",
    "Product Quality#Fall Protection",
    "Product Quality#Dustproof",
    "Product Quality#Water Resistant"
]


def show_examples(domain, label, n=3):
    subset = triplets_df[
        (triplets_df["domain"] == domain) &
        (triplets_df["aspect_category"] == label)
    ]

    print("\n" + "=" * 90)
    print(f"{domain.upper()} | {label}")
    print(f"Total annotations: {len(subset)}")
    print("=" * 90)

    if len(subset) == 0:
        print("No examples found.")
        return

    sample = subset[
        ["text", "aspect_term", "sentiment"]
    ].head(n)

    for i, row in enumerate(sample.itertuples(index=False), start=1):
        print(f"\nExample {i}")
        print("Text       :", row.text)
        print("Aspect term:", row.aspect_term)
        print("Sentiment  :", row.sentiment)


for label in phone_labels_to_inspect:
    show_examples("phone", label, n=3)


PHONE | Battery/Longevity#General
Total annotations: 84

Example 1
Text       : After half a year usage,I can confirm its a dodge spy phone! there's intermittent faults with it what definitely will cause your headaches!only good in this one the battery...so be a good mate to your self,and don't buy this Chinese s...t phone with built in UK spy software.... cheerio
Aspect term: battery
Sentiment  : positive

Example 2
Text       : Quick delivery. Battery works well but looks like recycled
Aspect term: Battery
Sentiment  : positive

Example 3
Text       : Quick delivery. Battery works well but looks like recycled
Aspect term: Battery
Sentiment  : negative

PHONE | Battery/Longevity#Power Consumption Speed
Total annotations: 32

Example 1
Text       : Redmi Note 10S is really very bad phone .battery drainage issues and always something is running in the background seems its uses your private information
Aspect term: battery
Sentiment  : negative

Example 2
Text       : I purchased the iP

In [64]:
def taxonomy_mapping_v02(domain, label):
    u = label.upper()

    # ========================================================
    # COMMON ASPECTS
    # ========================================================

    # ---------------- CHARGING / POWER ----------------
    # Put this before BATTERY because phone charging labels
    # start with Battery/Longevity.
    if domain == "phone" and u in {
        "BATTERY/LONGEVITY#CHARGING SPEED",
        "BATTERY/LONGEVITY#CHARGING METHOD",
    }:
        return "charging_power", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u.startswith("POWER_SUPPLY#"):
        return "charging_power", "COMMON", "yes", "confirmed"

    # ---------------- BATTERY ----------------
    if domain == "phone" and u in {
        "BATTERY/LONGEVITY#BATTERY CAPACITY",
        "BATTERY/LONGEVITY#BATTERY LIFE",
        "BATTERY/LONGEVITY#GENERAL",
        "BATTERY/LONGEVITY#POWER CONSUMPTION SPEED",
        "BATTERY/LONGEVITY#STANDBY TIME",
    }:
        return "battery", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u.startswith("BATTERY#"):
        return "battery", "COMMON", "yes", "confirmed"

    # ---------------- DISPLAY ----------------
    if domain == "phone" and u.startswith("SCREEN#"):
        return "display", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u.startswith("DISPLAY#"):
        return "display", "COMMON", "yes", "confirmed"

    # ---------------- PRICE ----------------
    if domain == "phone" and u.startswith("PRICE#"):
        return "price", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u == "LAPTOP#PRICE":
        return "price", "COMMON", "yes", "confirmed"

    # ---------------- PERFORMANCE ----------------
    if domain == "phone" and (
        u.startswith("PERFORMANCE#")
        or u == "SYSTEM#OPERATION SMOOTHNESS"
    ):
        return "performance", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u == "LAPTOP#OPERATION_PERFORMANCE":
        return "performance", "COMMON", "yes", "confirmed"

    # ---------------- DESIGN ----------------
    if domain == "phone" and u.startswith("APPEARANCE DESIGN#"):
        return "design", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u == "LAPTOP#DESIGN_FEATURES":
        return "design", "COMMON", "yes", "confirmed"

    # ---------------- CPU ----------------
    if domain == "phone" and u == "PRODUCT CONFIGURATION#CPU":
        return "cpu", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u.startswith("CPU#"):
        return "cpu", "COMMON", "yes", "confirmed"

    # ---------------- MEMORY ----------------
    if domain == "phone" and u in {
        "PRODUCT CONFIGURATION#MEMORY",
        "PRODUCT CONFIGURATION#OPERATING MEMORY",
    }:
        return "memory", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u.startswith("MEMORY#"):
        return "memory", "COMMON", "yes", "confirmed"

    # ---------------- USABILITY ----------------
    if domain == "phone" and u == "EASE OF USE#EASY TO USE":
        return "usability", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u == "LAPTOP#USABILITY":
        return "usability", "COMMON", "yes", "confirmed"

    # ---------------- SOFTWARE / SYSTEM ----------------
    if domain == "phone" and u in {
        "SYSTEM#SYSTEM GENERAL",
        "SYSTEM#APPLICATION",
        "SYSTEM#SOFTWARE COMPATIBILITY",
        "SYSTEM#SYSTEM UPGRADE",
        "SYSTEM#UI INTERFACE AESTHETICS",
        "SYSTEM#LOCK SCREEN DESIGN",
    }:
        return "software_system", "COMMON", "yes", "confirmed"

    if domain == "laptop" and (
        u.startswith("OS#")
        or u.startswith("SOFTWARE#")
    ):
        return "software_system", "COMMON", "yes", "confirmed"

    # ---------------- CONNECTIVITY ----------------
    if domain == "phone" and u in {
        "SMART CONNECT#BLUETOOTH CONNECTION",
        "SIGNAL#WIFI SIGNAL",
    }:
        return "connectivity", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u == "LAPTOP#CONNECTIVITY":
        return "connectivity", "COMMON", "yes", "confirmed"

    # ---------------- DURABILITY ----------------
    if domain == "phone" and u in {
        "PRODUCT QUALITY#FALL PROTECTION",
        "PRODUCT QUALITY#DUSTPROOF",
        "PRODUCT QUALITY#WATER RESISTANT",
    }:
        return "durability", "COMMON", "yes", "confirmed"

    if domain == "laptop" and u == "LAPTOP#QUALITY":
        return "durability", "COMMON", "yes", "confirmed"

    # ========================================================
    # PHONE-SPECIFIC ASPECTS
    # ========================================================

    if domain == "phone" and (
        u.startswith("CAMERA#")
        or u.startswith("SHOOTING FUNCTIONS#")
    ):
        return "camera", "PHONE_SPECIFIC", "yes", "confirmed"

    if domain == "phone" and u == "SIGNAL#CALL QUALITY":
        return "call_quality", "PHONE_SPECIFIC", "yes", "confirmed"

    if domain == "phone" and u in {
        "SIGNAL#SIGNAL GENERAL",
        "SIGNAL#SIGNAL OF MOBILE NETWORK",
    }:
        return "mobile_network", "PHONE_SPECIFIC", "yes", "confirmed"

    if domain == "phone" and u == "SECURITY#SCREEN UNLOCK":
        return "biometrics_security", "PHONE_SPECIFIC", "yes", "confirmed"

    if domain == "phone" and u == "SMART CONNECT#POSITIONING AND GPS":
        return "gps", "PHONE_SPECIFIC", "yes", "confirmed"

    if domain == "phone" and u == "SYSTEM#NFC":
        return "nfc", "PHONE_SPECIFIC", "yes", "confirmed"

    # ========================================================
    # LAPTOP-SPECIFIC ASPECTS
    # ========================================================

    if domain == "laptop" and u.startswith("KEYBOARD#"):
        return "keyboard", "LAPTOP_SPECIFIC", "yes", "confirmed"

    if domain == "laptop" and u.startswith("PORTS#"):
        return "ports", "LAPTOP_SPECIFIC", "yes", "confirmed"

    if domain == "laptop" and u.startswith("FANS&COOLING#"):
        return "fans_cooling", "LAPTOP_SPECIFIC", "yes", "confirmed"

    if domain == "laptop" and u.startswith("HARD_DISC#"):
        return "storage", "LAPTOP_SPECIFIC", "yes", "confirmed"

    if domain == "laptop" and u.startswith("MOUSE#"):
        return "mouse", "LAPTOP_SPECIFIC", "yes", "confirmed"

    if domain == "laptop" and u.startswith("MOTHERBOARD#"):
        return "motherboard", "LAPTOP_SPECIFIC", "yes", "confirmed"

    if domain == "laptop" and u.startswith("OPTICAL_DRIVES#"):
        return "optical_drive", "LAPTOP_SPECIFIC", "yes", "confirmed"

    if domain == "laptop" and u.startswith("GRAPHICS#"):
        return "graphics", "LAPTOP_SPECIFIC", "yes", "confirmed"

    if domain == "laptop" and u == "LAPTOP#PORTABILITY":
        return "portability", "LAPTOP_SPECIFIC", "yes", "confirmed"

    # ========================================================
    # CLEAR EXCLUSIONS
    # ========================================================

    # Phone non-product / service / accessory / noisy labels
    if domain == "phone" and (
        u == "OVERALL#OVERALL"
        or u.startswith("BUYER ATTITUDE#")
        or u.startswith("LOGISTICS#")
        or u.startswith("SELLER SERVICE#")
        or u.startswith("BRANDING/MARKETING#")
        or u.startswith("AFTER-SALES SERVICE#")
        or u.startswith("PRODUCT PACKAGING#")
        or u.startswith("PRODUCT ACCESSORIES#")
        or u == "EASE OF USE#AUDIENCE GROUPS"
        or u == "PRODUCT QUALITY#GENERAL"
        or u == "PRODUCT QUALITY#GENUINE PRODUCT"
        or u == "PRODUCT QUALITY#CLEANLINESS"
        or u.startswith("AUDIO/SOUND#")
        or u.startswith("KEY DESIGN#")
        or u.startswith("INTELLIGENT ASSISTANT#")
    ):
        return "", "", "no", "excluded"

    # Laptop generic / service / mixed-semantic labels
    if domain == "laptop" and (
        u == "LAPTOP#GENERAL"
        or u == "LAPTOP#MISCELLANEOUS"
        or u.startswith("OUT_OF_SCOPE#")
        or u.startswith("SHIPPING#")
        or u.startswith("SUPPORT#")
        or u.startswith("COMPANY#")
        or u.startswith("WARRANTY#")
        or u.startswith("MULTIMEDIA_DEVICES#")
        or u.startswith("HARDWARE#")
    ):
        return "", "", "no", "excluded"

    # Anything not resolved above still needs inspection
    return "", "", "review", "pending"

In [65]:
mapping_v02 = mapping_df.copy()

mapped_v02 = mapping_v02.apply(
    lambda row: taxonomy_mapping_v02(
        row["domain"],
        row["source_label"]
    ),
    axis=1
)

mapping_v02[
    [
        "canonical_aspect",
        "semantic_group",
        "include",
        "mapping_status"
    ]
] = pd.DataFrame(
    mapped_v02.tolist(),
    index=mapping_v02.index
)

mapping_v02.head(20)

,domain,source_label,count,canonical_aspect,semantic_group,include,mapping_status
0,phone,Overall#Overall,910,,,no,excluded
1,phone,Screen#Clarity,22,display,COMMON,yes,confirmed
2,phone,Screen#Size,35,display,COMMON,yes,confirmed
3,phone,Product Configuration#Operating Memory,35,memory,COMMON,yes,confirmed
4,phone,Product Accessories#Cell Phone Film,46,,,no,excluded
5,phone,Product Accessories#Phone Cases,69,,,no,excluded
6,phone,Price#Price,150,price,COMMON,yes,confirmed
7,phone,Camera#General,145,camera,PHONE_SPECIFIC,yes,confirmed
8,phone,Battery/Longevity#Battery Life,105,battery,COMMON,yes,confirmed
9,phone,System#Software Compatibility,30,software_system,COMMON,yes,confirmed


In [66]:
print("=" * 75)
print("1. V0.2 INCLUDE STATUS")
print("=" * 75)

print(
    mapping_v02.groupby(
        ["domain", "include"]
    ).size()
)


print("\n" + "=" * 75)
print("2. V0.2 SEMANTIC GROUP SUMMARY")
print("=" * 75)

print(
    mapping_v02[
        mapping_v02["include"] == "yes"
    ]
    .groupby(
        ["domain", "semantic_group"]
    )
    .size()
)


print("\n" + "=" * 75)
print("3. REMAINING REVIEW LABELS")
print("=" * 75)

remaining_review = mapping_v02[
    mapping_v02["include"] == "review"
][
    [
        "domain",
        "source_label",
        "count",
        "canonical_aspect",
        "semantic_group",
        "mapping_status"
    ]
]

if len(remaining_review) == 0:
    print("No unresolved labels.")
else:
    print(remaining_review.to_string(index=False))

1. V0.2 INCLUDE STATUS
domain  include
laptop  no         29
        yes        79
phone   no         35
        yes        51
dtype: int64

2. V0.2 SEMANTIC GROUP SUMMARY
domain  semantic_group 
laptop  COMMON             43
        LAPTOP_SPECIFIC    36
phone   COMMON             39
        PHONE_SPECIFIC     12
dtype: int64

3. REMAINING REVIEW LABELS
No unresolved labels.


In [67]:
v02_path = (
    PROJECT_ROOT
    / "taxonomy"
    / "mabsa_taxonomy_mapping_v0_2.csv"
)

mapping_v02.to_csv(
    v02_path,
    index=False
)

print("Saved:")
print(v02_path.resolve())

Saved:
D:\projects\J26-DS-349\component_1\taxonomy\mabsa_taxonomy_mapping_v0_2.csv


In [70]:
# ============================================================
# V0.2 TAXONOMY VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. Basic mapping integrity checks
# ------------------------------------------------------------

duplicate_mappings = mapping_v02.duplicated(
    subset=["domain", "source_label"]
).sum()

included_missing_aspect = (
    (mapping_v02["include"] == "yes") &
    (mapping_v02["canonical_aspect"].astype(str).str.strip() == "")
).sum()

included_missing_group = (
    (mapping_v02["include"] == "yes") &
    (mapping_v02["semantic_group"].astype(str).str.strip() == "")
).sum()

unresolved_labels = (
    mapping_v02["include"] == "review"
).sum()

print("=" * 80)
print("1. MAPPING INTEGRITY")
print("=" * 80)

print("Duplicate domain + source-label mappings :", duplicate_mappings)
print("Included labels missing canonical aspect :", included_missing_aspect)
print("Included labels missing semantic group   :", included_missing_group)
print("Remaining review labels                  :", unresolved_labels)


# ------------------------------------------------------------
# 2. Merge taxonomy with all annotated triplets
# ------------------------------------------------------------

mapped_triplets = triplets_df.merge(
    mapping_v02[
        [
            "domain",
            "source_label",
            "canonical_aspect",
            "semantic_group",
            "include",
            "mapping_status"
        ]
    ],
    left_on=["domain", "aspect_category"],
    right_on=["domain", "source_label"],
    how="left",
    validate="many_to_one"
)

missing_after_merge = mapped_triplets["include"].isna().sum()

print("\n" + "=" * 80)
print("2. MERGE VALIDATION")
print("=" * 80)

print("Total annotated triplets :", len(mapped_triplets))
print("Unmapped after merge     :", missing_after_merge)


# ------------------------------------------------------------
# 3. Annotation-level include/exclude coverage
# ------------------------------------------------------------

annotation_coverage = (
    mapped_triplets
    .groupby(["domain", "include"])
    .size()
    .reset_index(name="annotation_count")
)

domain_totals = (
    mapped_triplets
    .groupby("domain")
    .size()
    .rename("domain_total")
    .reset_index()
)

annotation_coverage = annotation_coverage.merge(
    domain_totals,
    on="domain",
    how="left"
)

annotation_coverage["percentage"] = (
    annotation_coverage["annotation_count"]
    / annotation_coverage["domain_total"]
    * 100
).round(2)

print("\n" + "=" * 80)
print("3. ANNOTATION-LEVEL COVERAGE")
print("=" * 80)

print(annotation_coverage.to_string(index=False))


# ------------------------------------------------------------
# 4. Semantic-group distribution at annotation level
# ------------------------------------------------------------

semantic_group_annotations = (
    mapped_triplets[
        mapped_triplets["include"] == "yes"
    ]
    .groupby(["domain", "semantic_group"])
    .size()
    .reset_index(name="annotation_count")
)

print("\n" + "=" * 80)
print("4. SEMANTIC GROUP ANNOTATION COUNTS")
print("=" * 80)

print(semantic_group_annotations.to_string(index=False))


# ------------------------------------------------------------
# 5. Canonical-aspect distribution
# ------------------------------------------------------------

canonical_aspect_counts = (
    mapped_triplets[
        mapped_triplets["include"] == "yes"
    ]
    .groupby(
        ["domain", "semantic_group", "canonical_aspect"]
    )
    .size()
    .reset_index(name="annotation_count")
    .sort_values(
        ["domain", "annotation_count"],
        ascending=[True, False]
    )
)

print("\n" + "=" * 80)
print("5. CANONICAL ASPECT ANNOTATION COUNTS")
print("=" * 80)

print(canonical_aspect_counts.to_string(index=False))

1. MAPPING INTEGRITY
Duplicate domain + source-label mappings : 0
Included labels missing canonical aspect : 0
Included labels missing semantic group   : 0
Remaining review labels                  : 0

2. MERGE VALIDATION
Total annotated triplets : 7765
Unmapped after merge     : 0

3. ANNOTATION-LEVEL COVERAGE
domain include  annotation_count  domain_total  percentage
laptop      no               965          2955       32.66
laptop     yes              1990          2955       67.34
 phone      no              3058          4810       63.58
 phone     yes              1752          4810       36.42

4. SEMANTIC GROUP ANNOTATION COUNTS
domain  semantic_group  annotation_count
laptop          COMMON              1595
laptop LAPTOP_SPECIFIC               395
 phone          COMMON              1434
 phone  PHONE_SPECIFIC               318

5. CANONICAL ASPECT ANNOTATION COUNTS
domain  semantic_group    canonical_aspect  annotation_count
laptop          COMMON         performance        

In [71]:
annotation_coverage.to_csv(
    RESULTS_DIR / "taxonomy_annotation_coverage_v0_2.csv",
    index=False
)

semantic_group_annotations.to_csv(
    RESULTS_DIR / "taxonomy_semantic_group_counts_v0_2.csv",
    index=False
)

canonical_aspect_counts.to_csv(
    RESULTS_DIR / "taxonomy_canonical_aspect_counts_v0_2.csv",
    index=False
)

mapped_triplets.to_csv(
    RESULTS_DIR / "mapped_triplets_v0_2.csv",
    index=False
)

print("V0.2 taxonomy validation outputs saved successfully.")

V0.2 taxonomy validation outputs saved successfully.
